# 2. The live stream, and a mock extractor from start to results

An extractor doesn't get a finished transcript. It gets what a streaming speech-to-text service sends during a conversation: **partials** (the current turn so far, replaced about every second) and **finished turns**.

This notebook shows that stream, runs a small extractor on it, and follows its output to the scores. `docs/stream.md` and `docs/interface.md` have the details.

In [ ]:
import json
import statistics
from itertools import pairwise

import pandas as pd

from live_form_eval import Extractor, Session, evaluate, load_config, run
from live_form_eval.data.common.store import load_dataset
from live_form_eval.harness.stream import build_stream

pd.set_option('display.max_colwidth', 110)
config = load_config()
dev = load_dataset(config, 'spokenwoz', split='dev', n=40)
call_id = min(dev.conversations)
call = dev.conversations[call_id]

## What an extractor receives

Each row is one message, at the moment it arrives (`t`, seconds since the call started).

- A `partial` is the whole turn so far: it **replaces** the previous partial of that turn.
- A `turn` is the finished turn. A turn ends at a pause after a sentence, not when the speaker changes, so one turn can hold two speakers.
- `speaker` is the turn's main speaker (`A` = the first to speak: the caller). `PENDING` means under a second of audio.

In [ ]:
stream = build_stream(call.words, config.stream, key=call.id)
pd.DataFrame([{'t': p.t, 'kind': p.kind, 'turn_id': p.turn_id, 'speaker': p.speaker, 'labelled_text()': p.labelled_text()}
              for p in stream[:16]])

A finished turn also carries its words, each with its audio time and its own speaker. A partial's words have no speaker: the service labels only final words.

In [ ]:
finished = next(p for p in stream if p.kind == 'turn' and len({w.speaker for w in p.words}) > 1)
print(f'turn {finished.turn_id}, arrived at {finished.t} s, main speaker {finished.speaker}')
pd.DataFrame([word.model_dump() for word in finished.words]).T

### How close is this to a real service?

The stream is simulated from the datasets' word times, because no dataset has partials. Its timing was measured on real AssemblyAI streams (`docs/stream.md`). The same statistics, on 40 simulated calls:

In [ ]:
gaps, lags, lengths, shared = [], [], [], []
for conversation in dev.conversations.values():
    first_seen, by_turn = {}, {}
    for p in build_stream(conversation.words, config.stream, key=conversation.id):
        by_turn.setdefault(p.turn_id, []).append(p)
        for word in p.words:
            first_seen.setdefault((word.start, word.end, word.text), p.t)
    lags += [t - key[1] for key, t in first_seen.items()]
    for messages in by_turn.values():
        partials = [p.t for p in messages if p.kind == 'partial']
        gaps += [b - a for a, b in pairwise(partials)]
        words = messages[-1].words
        lengths.append(max(w.end for w in words) - min(w.start for w in words))
        shared.append(len({w.speaker for w in words} - {'PENDING'}) > 1)
pd.DataFrame([
    {'measure': 'time between two partials of a turn, median', 'simulated': f'{statistics.median(gaps):.2f} s', 'real streams': '1.21–1.40 s'},
    {'measure': 'a word spoken → first shown, median', 'simulated': f'{statistics.median(lags):.2f} s', 'real streams': '0.75 s (dialogues), 1.06 s (monologues)'},
    {'measure': 'a word spoken → first shown, 90th percentile', 'simulated': f'{sorted(lags)[int(0.9 * len(lags))]:.2f} s', 'real streams': '1.6–1.75 s'},
    {'measure': 'turn length, median', 'simulated': f'{statistics.median(lengths):.1f} s', 'real streams': '3.9 s'},
    {'measure': 'turns that hold two speakers', 'simulated': f'{sum(shared) / len(shared):.0%}', 'real streams': 'frequent (not counted)'},
])

## A mock extractor

An extractor is a class with an `async run(session)` method. It reads `session.partials()` and publishes values by calling the session. Nothing is returned: the harness records every call with the conversation's clock.

This one uses no model. It fills two fields from keywords, to show the mechanics: the train day when a weekday is said in a turn about a train, and the number of train tickets.

In [ ]:
import re

DAYS = ('monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday')
NUMBERS = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8}


class KeywordExtractor(Extractor):
    async def run(self, session: Session) -> None:
        shown = {}                                    # this conversation's state lives inside run()
        async for p in session.partials():            # each partial or finished turn, when it arrives
            text = p.text.lower()
            values = {}
            if 'train' in text:
                day = next((d for d in DAYS if d in text), None)
                if day:
                    values['train_day'] = day         # a select field: exactly one of its options
                tickets = re.search(r'(\w+) (?:tickets?|people)', text)
                if tickets and tickets.group(1) in NUMBERS:
                    values['train_people'] = NUMBERS[tickets.group(1)]   # an integer field: an int
            values = {field: value for field, value in values.items() if shown.get(field) != value}
            if values:
                session.update_snapshot(values)       # on screen from this moment
                shown.update(values)
            # a real extractor reports each model call: its units (tokens, …) and their prices in euros
            session.report_usage(model='keywords', extraction=True)

### Run it

`run()` replays the conversations as live streams and writes a run folder. Here: 10 dev calls at 30 times real time, which makes the run **unofficial**. An official run uses the config's sample at real time (`lfe run … --dataset spokenwoz`).

In [ ]:
run_dir = run(KeywordExtractor(), 'spokenwoz', config=config, split='dev', n=10, speed=30, concurrency=10)
sorted(path.name for path in run_dir.iterdir())

### The run folder

`predictions.jsonl` has one line per published value, `calls.jsonl` one line per reported use (a model call, with its units and their prices), and `run.json` describes the run. The evaluation reads only these files and the gold.

In [ ]:
print('predictions.jsonl')
print('\n'.join((run_dir / 'predictions.jsonl').read_text().splitlines()[:5]))
print('\ncalls.jsonl')
print('\n'.join((run_dir / 'calls.jsonl').read_text().splitlines()[:2]))
info = json.loads((run_dir / 'run.json').read_text())
print('\nrun.json:', {key: info[key] for key in ('run_id', 'extractor', 'dataset', 'split', 'speed', 'official')})
print('first conversation:', info['conversations'][0])

### From the run folder to the scores

`evaluate()` rebuilds what each field showed over time and compares it with the gold. The keyword extractor fills 2 of 37 fields: what it shows is right (precision is high), but most values are never shown, so completeness, speed and outcome are low, and so is the score.

In [ ]:
result = evaluate(run_dir, config=config, write=False)
print(result.summary_text())
result.per_field[result.per_field['field_id'].isin(['train_day', 'train_people'])][
    ['field_id', 'correct', 'wrong', 'missing', 'invented', 'final_f1', 'completeness', 'precision', 'speed']]

## How the scores react: an oracle with known errors

`examples/oracle_extractor.py` reads the gold and publishes each value a fixed delay after the stream delivered its words. It is not a submission (it cheats): it shows what each score measures.

- **Oracle:** every value 1 s after its words arrive in the stream.
- **SloppyOracle:** 2 s late, 10% of the values missing, 5% wrong, a few invented.

In [ ]:
import sys

from live_form_eval.config import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT))
from examples.oracle_extractor import Oracle, SloppyOracle

runs = [run_dir] + [run(extractor(), 'spokenwoz', config=config, split='dev', n=10, speed=30, concurrency=10)
                    for extractor in (Oracle, SloppyOracle)]
comparison = evaluate(runs, config=config, write=False)
latency = f'latency_p{config.latency.percentile:g}_s'
comparison.summary.set_index('name')[['score', 'completeness', 'precision', 'speed', 'outcome', 'final_f1',
                                      'fixes_per_form', 'latency_median_s', latency, 'never_settled', 'jga']].T

Reading the table:

- The **score** is the harmonic mean of the four rows below it, so it stays close to the weakest one.
- The oracle shows every value, and nothing else: **completeness**, **precision** and **outcome** are 1. Its **speed** is what a 1 s delay is worth on the speed curve, about 0.93. Every clock starts when the words reach the extractor, so the time the stream takes to deliver them is not counted.
- The sloppy oracle misses values (completeness and outcome go down), shows some wrong ones (precision goes down) and is 2 s late (speed goes down).
- **Latency** is reported next to the score, not ranked. Values that never appear never settle, so the sloppy oracle's 90th percentile is infinite.

Notebook 3 goes through every table and plot of an evaluation.